# 05 — RAG schema-aware pour la génération AQL

Objectif : construire proprement le module RAG demandé par le cahier des charges.

Pipeline de ce notebook :

**Question NL → LangChain → ChromaDB → SentenceTransformers → schéma + exemples pertinents → Prompt RAG → LLM Ollama → AQL → validation → confidentialité → exécution contrôlée**


In [1]:
%pip install -q langchain langchain-core langchain-chroma langchain-huggingface chromadb sentence-transformers python-arango pandas
# À exécuter une seule fois dans un nouvel environnement

Note: you may need to restart the kernel to use updated packages.


In [1]:
import json
import sys
from pathlib import Path

import chromadb
import pandas as pd

from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# ------------------------------------------------------------
# Détection robuste de la racine du projet
# ------------------------------------------------------------
current = Path.cwd().resolve()

candidates = [
    current,
    current / "LLM_Yelp_Project",
    current.parent,
    Path("/kaggle/working/LLM_Yelp_Project"),
]

PROJECT_ROOT = None
for candidate in candidates:
    if (candidate / "arangodb" / "schema_description.json").exists() and \
       (candidate / "data" / "benchmark" / "yelp_benchmark.json").exists():
        PROJECT_ROOT = candidate.resolve()
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Racine du projet introuvable. Placez le notebook dans le projet "
        "ou adaptez PROJECT_ROOT."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SCHEMA_PATH = PROJECT_ROOT / "arangodb" / "schema_description.json"
BENCHMARK_PATH = PROJECT_ROOT / "data" / "benchmark" / "yelp_benchmark.json"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("SCHEMA_PATH  :", SCHEMA_PATH)
print("BENCHMARK    :", BENCHMARK_PATH)

PROJECT_ROOT : C:\Users\hp\Desktop\bureau\s8\LLM_Yelp_Project
SCHEMA_PATH  : C:\Users\hp\Desktop\bureau\s8\LLM_Yelp_Project\arangodb\schema_description.json
BENCHMARK    : C:\Users\hp\Desktop\bureau\s8\LLM_Yelp_Project\data\benchmark\yelp_benchmark.json


In [2]:
from src.prompt_builder import PromptBuilder
from src.query_generator import QueryGenerator
from src.query_validator import QueryValidator
from src.privacy_filter import PrivacyFilter
from src.query_executor import QueryExecutor
from src.db_connector import get_database

with open(SCHEMA_PATH, "r", encoding="utf-8") as f:
    schema = json.load(f)

with open(BENCHMARK_PATH, "r", encoding="utf-8") as f:
    benchmark = json.load(f)

assert len(benchmark) == 128, f"128 questions attendues, obtenu : {len(benchmark)}"
assert all(item.get("id") and item.get("question") and item.get("gold_aql") for item in benchmark)

print("Base :", schema.get("database"))
print("Graphe :", schema.get("graph"))
print("Questions benchmark :", len(benchmark))
print("Collections document :", len(schema["document_collections"]))
print("Collections edge :", len(schema["edge_collections"]))

Base : YelpDB
Graphe : YelpGraph
Questions benchmark : 128
Collections document : 7
Collections edge : 7


## 1. Transformer le schéma réel en documents RAG

Aucun schéma n'est écrit manuellement ici : les collections, attributs et relations proviennent directement de `arangodb/schema_description.json`.

In [3]:
def build_schema_documents(schema_dict):
    documents = []

    for name, info in schema_dict["document_collections"].items():
        fields = list(info.get("fields", {}).keys())
        text = (
            f"Document collection: {name}\n"
            f"Description: {info.get('description', '')}\n"
            f"Fields: {', '.join(fields)}"
        )
        documents.append(
            Document(
                page_content=text,
                metadata={"kind": "document", "name": name}
            )
        )

    for name, info in schema_dict["edge_collections"].items():
        from_collection = info.get("from", "")
        to_collection = info.get("to", "")
        text = (
            f"Edge collection: {name}\n"
            f"Description: {info.get('description', '')}\n"
            f"Relation: {from_collection} -> {to_collection}"
        )
        documents.append(
            Document(
                page_content=text,
                metadata={
                    "kind": "edge",
                    "name": name,
                    "from": from_collection,
                    "to": to_collection,
                }
            )
        )

    return documents


schema_documents = build_schema_documents(schema)
print("Documents de schéma :", len(schema_documents))
for doc in schema_documents[:2]:
    print("\n", doc.page_content)

Documents de schéma : 14

 Document collection: Businesses
Description: Businesses from the Yelp dataset.
Fields: bid, business_id, name, full_address, city, latitude, longitude, review_count, is_open, rating, state

 Document collection: Users
Description: Users from the Yelp dataset.
Fields: uid, user_id, name


## 2. Transformer les 128 exemples Question + Gold AQL en documents

Le Gold est utilisé uniquement comme **exemple few-shot récupérable pour les autres questions**. Lorsqu'on traite `Qxxx`, son propre exemple `Qxxx` est retiré des résultats RAG.

In [4]:
def build_example_documents(items):
    documents = []

    for item in items:
        collections = ",".join(item.get("collections", []))
        text = (
            f"Question:\n{item['question']}\n\n"
            f"Gold AQL:\n{item['gold_aql'].strip()}"
        )
        documents.append(
            Document(
                page_content=text,
                metadata={
                    "question_id": item["id"],
                    "difficulty": item.get("difficulty", ""),
                    "type": item.get("type", ""),
                    "collections": collections,
                }
            )
        )

    return documents


example_documents = build_example_documents(benchmark)
print("Exemples préparés :", len(example_documents))
print(example_documents[0].metadata)

Exemples préparés : 128
{'question_id': 'Q001', 'difficulty': 'medium', 'type': 'graph_filter', 'collections': 'Businesses,BusinessCategory,Categories'}


## 3. SentenceTransformers + ChromaDB + LangChain

`HuggingFaceEmbeddings` de LangChain utilise ici le modèle SentenceTransformers `all-MiniLM-L6-v2`. ChromaDB assure l'indexation vectorielle.

In [5]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    encode_kwargs={"normalize_embeddings": True},
)

# Base Chroma en mémoire : propre à chaque exécution du notebook
chroma_client = chromadb.EphemeralClient()

schema_store = Chroma.from_documents(
    documents=schema_documents,
    embedding=embeddings,
    collection_name="yelp_schema",
    client=chroma_client,
)

examples_store = Chroma.from_documents(
    documents=example_documents,
    embedding=embeddings,
    collection_name="yelp_examples",
    client=chroma_client,
)

print("Schéma indexé   :", schema_store._collection.count())
print("Exemples indexés:", examples_store._collection.count())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Schéma indexé   : 14
Exemples indexés: 128


## 4. Retrieval sans fuite du Gold

Le retrieval combine :
1. les exemples les plus proches sémantiquement ;
2. les collections citées par ces exemples ;
3. les éléments du schéma proches directement de la question ;
4. les collections source/cible nécessaires lorsqu'une relation edge est retenue.

In [7]:
def retrieve_examples(question, current_question_id, top_k=3):
    # On demande plus de candidats afin de pouvoir exclure la question courante.
    candidates = examples_store.similarity_search_with_score(
        question,
        k=min(top_k + 10, len(benchmark))
    )

    selected = []
    for doc, score in candidates:
        if doc.metadata.get("question_id") == current_question_id:
            continue
        selected.append((doc, score))
        if len(selected) == top_k:
            break

    return selected


def retrieve_schema(question, retrieved_examples, semantic_k=4):
    selected_by_name = {}

    # A. Schéma sémantiquement proche de la question
    for doc, score in schema_store.similarity_search_with_score(question, k=semantic_k):
        selected_by_name[doc.metadata["name"]] = doc

    # B. Collections présentes dans les exemples similaires
    wanted_names = set()
    for doc, _ in retrieved_examples:
        raw = doc.metadata.get("collections", "")
        wanted_names.update(x.strip() for x in raw.split(",") if x.strip())

    for doc in schema_documents:
        if doc.metadata["name"] in wanted_names:
            selected_by_name[doc.metadata["name"]] = doc

    # C. Si une relation est retenue, inclure ses extrémités
    edge_names = [
        name for name, doc in selected_by_name.items()
        if doc.metadata.get("kind") == "edge"
    ]
    for edge_name in edge_names:
        edge_info = schema["edge_collections"].get(edge_name, {})
        wanted_names.update([edge_info.get("from", ""), edge_info.get("to", "")])

    for doc in schema_documents:
        if doc.metadata["name"] in wanted_names:
            selected_by_name[doc.metadata["name"]] = doc

    return list(selected_by_name.values())


def format_examples(retrieved_examples):
    return "\n\n---\n\n".join(doc.page_content for doc, _ in retrieved_examples)


def format_schema(retrieved_schema):
    return "\n\n".join(doc.page_content for doc in retrieved_schema)

## 5. Chaîne LangChain RAG

La chaîne prend `{question_id, question}` et produit le contexte RAG ainsi que le prompt final utilisé par `QueryGenerator`.

In [8]:
def retrieve_payload(payload):
    retrieved_examples = retrieve_examples(
        question=payload["question"],
        current_question_id=payload["question_id"],
        top_k=payload.get("top_k_examples", 3),
    )

    retrieved_schema = retrieve_schema(
        question=payload["question"],
        retrieved_examples=retrieved_examples,
        semantic_k=payload.get("top_k_schema", 4),
    )

    return {
        **payload,
        "retrieved_examples": retrieved_examples,
        "retrieved_schema": retrieved_schema,
        "examples_context": format_examples(retrieved_examples),
        "schema_context": format_schema(retrieved_schema),
    }


def build_prompt_payload(payload):
    prompt = PromptBuilder.build_rag_prompt(
        question=payload["question"],
        schema_context=payload["schema_context"],
        examples_context=payload["examples_context"],
    )
    return {**payload, "prompt": prompt}


rag_chain = RunnableLambda(retrieve_payload) | RunnableLambda(build_prompt_payload)
print("Chaîne LangChain créée.")

Chaîne LangChain créée.


## 6. Test de retrieval sur Q001

Le test ci-dessous vérifie explicitement que `Q001` n'apparaît pas dans les exemples fournis au LLM lorsqu'on génère Q001.

In [9]:
test_item = benchmark[0]

rag_input = rag_chain.invoke({
    "question_id": test_item["id"],
    "question": test_item["question"],
    "top_k_examples": 3,
    "top_k_schema": 4,
})

retrieved_ids = [
    doc.metadata["question_id"]
    for doc, _ in rag_input["retrieved_examples"]
]

assert test_item["id"] not in retrieved_ids, "Fuite du Gold détectée !"

print("Question :", test_item["question"])
print("Exemples récupérés :", retrieved_ids)
print("Schéma récupéré :", [d.metadata["name"] for d in rag_input["retrieved_schema"]])
print("Fuite du Gold : NON")

Question : Give me all the moroccan restaurants in Texas
Exemples récupérés : ['Q101', 'Q056', 'Q075']
Schéma récupéré : ['Neighborhoods', 'Businesses', 'Checkins', 'Tips', 'Categories', 'BusinessCategory']
Fuite du Gold : NON


In [13]:
current_question = test_item["question"]

schema_context = rag_input["schema_context"]
examples_context = rag_input["examples_context"]

rag_prompt = PromptBuilder.build_rag_prompt(
    question=current_question,
    schema_context=schema_context,
    examples_context=examples_context
)

print("Question :", current_question)
print("\nPrompt RAG créé avec succès.")

Question : Give me all the moroccan restaurants in Texas

Prompt RAG créé avec succès.


In [10]:
print("===== CONTEXTE SCHÉMA =====")
print(rag_input["schema_context"])

print("\n===== EXEMPLES RÉCUPÉRÉS =====")
print(rag_input["examples_context"])

===== CONTEXTE SCHÉMA =====
Document collection: Neighborhoods
Description: Neighborhood information associated with businesses.
Fields: id, business_id, neighborhood_name

Document collection: Businesses
Description: Businesses from the Yelp dataset.
Fields: bid, business_id, name, full_address, city, latitude, longitude, review_count, is_open, rating, state

Document collection: Checkins
Description: Check-in information associated with businesses.
Fields: cid, business_id, count, day

Document collection: Tips
Description: Tips written by users about businesses.
Fields: tip_id, business_id, text, user_id, likes, year, month

Document collection: Categories
Description: Categories associated with businesses.
Fields: id, business_id, category_name

Edge collection: BusinessCategory
Description: A business is associated with a category.
Relation: Businesses -> Categories

===== EXEMPLES RÉCUPÉRÉS =====
Question:
Find all cities in Texas in which there is a restaurant callled MGM Grand 

## 7. Génération AQL RAG

Pour le notebook 05, un seul modèle suffit pour démontrer le fonctionnement du RAG. La comparaison finale de deux LLM sur les 128 questions sera faite dans le notebook 06.

In [11]:
DEMO_MODEL = "mistral:latest"

generator = QueryGenerator(model_name=DEMO_MODEL)
validator = QueryValidator(SCHEMA_PATH)
privacy_filter = PrivacyFilter()

generation = generator.generate(rag_input["prompt"])
generated_aql = generation["aql"]
validation = validator.validate(generated_aql)
privacy = privacy_filter.check(generated_aql)

print("Modèle :", generation["model"])
print("Temps :", round(generation["generation_time"], 2), "s")
print("\nAQL généré :\n", generated_aql)
print("\nValidation :", validation)
print("\nConfidentialité :", privacy)

Modèle : mistral:latest
Temps : 130.78 s

AQL généré :
 FOR b IN Businesses
FILTER b.name CONTAINS "Moroccan"
FILTER b.state == "Texas"
RETURN b._id

Validation : {'valid': True, 'errors': [], 'warnings': ['No LIMIT detected. A LIMIT is recommended for queries that may return many documents.'], 'used_collections': ['Businesses'], 'invalid_attributes': []}

Confidentialité : {'allowed': True, 'sensitive_fields': [], 'reason': None}


## 8. Pipeline complet avec correction agentique optionnelle

Le cahier prévoit également une approche **agent avec validation/correction**. Le module `QueryGenerator` du projet contient déjà `generate_with_correction()`. Ici, on l'utilise avec une seule correction LLM et **sans réparation déterministe spécifique**, afin que l'expérience reste générale.

In [16]:
from src.db_connector import get_database
from src.query_executor import QueryExecutor
# Connexion à ArangoDB
db = get_database(
    database="YelpDB"
)

# Exécuteur contrôlé
executor = QueryExecutor(
    db,
    validator
)

print("Connexion ArangoDB et QueryExecutor prêts.")

Connexion ArangoDB et QueryExecutor prêts.


In [17]:
agent_generation = generator.generate_with_correction(
    prompt=rag_prompt,
    question=current_question,
    validator=validator,
    schema_context=schema_context,
    examples_context=examples_context,
    max_corrections=1,
    use_schema_repair=True,
    executor=executor,
    privacy_filter=privacy_filter,
    max_execution_corrections=1
)

## 9. Exécution contrôlée dans ArangoDB

Cette cellule doit être exécutée sur la machine où `YelpDB` est accessible. Une requête n'est exécutée que si elle passe le filtre de confidentialité ; `QueryExecutor` refait ensuite la validation avant l'exécution.

In [18]:
print("=" * 80)
print("AQL ORIGINAL")
print("=" * 80)
print(agent_generation["original_aql"])

print("\n" + "=" * 80)
print("AQL FINAL")
print("=" * 80)
print(agent_generation["aql"])

print("\nValidation :")
print(agent_generation["validation"])

print("\nPrivacy :")
print(agent_generation["privacy"])

print(
    "\nCorrection validation :",
    agent_generation["correction_attempts"]
)

print(
    "Correction après erreur ArangoDB :",
    agent_generation["execution_correction_attempts"]
)

execution = agent_generation["execution"]

if execution is not None:
    print("\nExécutée :", execution["executed"])
    print("Erreur :", execution.get("error"))

    if execution["executed"]:
        results = execution.get("results", [])
        print("Nombre de résultats :", len(results))
        print("Aperçu :", results[:5])

AQL ORIGINAL
FOR b IN Businesses
    FILTER b.state == "Texas"
    LET categories = (
        FOR c IN 1..1 OUTBOUND b BusinessCategory
            RETURN c.category_name
    )
    FILTER "Moroccan" IN categories
    RETURN b

AQL FINAL
FOR b IN Businesses
    FILTER b.state == "Texas"
    LET categories = (
        FOR c IN 1..1 OUTBOUND b BusinessCategory
            RETURN c.category_name
    )
    FILTER "Moroccan" IN categories
    RETURN b

Validation :
{'valid': True, 'errors': [], 'warnings': ['No LIMIT detected. A LIMIT is recommended for queries that may return many documents.'], 'used_collections': ['BusinessCategory', 'Businesses'], 'invalid_attributes': []}

Privacy :
{'allowed': True, 'sensitive_fields': [], 'reason': None}

Correction validation : 0
Correction après erreur ArangoDB : 0

Exécutée : True
Erreur : None
Nombre de résultats : 0
Aperçu : []


### Conclusion du notebook 05

Le module RAG utilise maintenant explicitement **LangChain + ChromaDB + SentenceTransformers**, le schéma réel d'ArangoDB et les exemples Gold du benchmark. La question courante est exclue du retrieval des exemples pour éviter une fuite de réponse. Le notebook démontre aussi la génération, la validation, le contrôle de confidentialité, la correction agentique et l'exécution contrôlée.

In [19]:
test_item = benchmark[0]

print("ID :", test_item["id"])
print("Question :", test_item["question"])
print("\nGOLD AQL :")
print(test_item["gold_aql"])

ID : Q001
Question : Give me all the moroccan restaurants in Texas

GOLD AQL :

FOR b IN Businesses
    FILTER b.state == "Texas"

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Moroccan"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            city: b.city,
            state: b.state,
            rating: b.rating
        }



In [20]:
gold_aql = test_item["gold_aql"]

try:
    gold_results = list(
        db.aql.execute(gold_aql)
    )

    print("Gold exécuté : True")
    print("Nombre de résultats :", len(gold_results))
    print("Résultats :", gold_results[:5])

except Exception as e:
    print("Gold exécuté : False")
    print("Erreur :", e)

Gold exécuté : True
Nombre de résultats : 0
Résultats : []


In [21]:
test_item = benchmark[1]

print("ID :", test_item["id"])
print("Question :", test_item["question"])
print("\nGOLD AQL :")
print(test_item["gold_aql"])

ID : Q002
Question : List all the Italian restaurants in Los Angeles

GOLD AQL :

FOR b IN Businesses
    FILTER b.city == "Los Angeles"

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Italian"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            city: b.city,
            rating: b.rating
        }



In [22]:
gold_aql = test_item["gold_aql"]

try:
    gold_results = list(
        db.aql.execute(gold_aql)
    )

    print("Gold exécuté : True")
    print("Nombre de résultats :", len(gold_results))
    print("Aperçu :", gold_results[:5])

except Exception as e:
    print("Gold exécuté : False")
    print("Erreur :", e)

Gold exécuté : True
Nombre de résultats : 0
Aperçu : []


In [23]:
# Q002 est déjà dans test_item

current_question = test_item["question"]

# RAG
rag_input = rag_chain.invoke({
    "question_id": test_item["id"],
    "question": current_question,
    "top_k_examples": 3,
    "top_k_schema": 4,
})

schema_context = rag_input["schema_context"]
examples_context = rag_input["examples_context"]

# Prompt
rag_prompt = PromptBuilder.build_rag_prompt(
    question=current_question,
    schema_context=schema_context,
    examples_context=examples_context
)

# Génération du modèle
agent_generation = generator.generate_with_correction(
    prompt=rag_prompt,
    question=current_question,
    validator=validator,
    schema_context=schema_context,
    examples_context=examples_context,
    max_corrections=1,
    use_schema_repair=True,
    executor=executor,
    privacy_filter=privacy_filter,
    max_execution_corrections=1
)

print("GENERATED AQL :")
print(agent_generation["aql"])

generated_execution = agent_generation["execution"]

print("\nExécutée :", generated_execution["executed"])
print("Erreur :", generated_execution.get("error"))

if generated_execution["executed"]:
    generated_results = generated_execution["results"]

    print(
        "Nombre de résultats GENERATED :",
        len(generated_results)
    )

    print(
        "Aperçu GENERATED :",
        generated_results[:5]
    )

GENERATED AQL :
FOR b IN Businesses
FILTER b.city == "Los Angeles"
FILTER "Italian" IN (
    FOR c IN 1..1 OUTBOUND b BusinessCategory
    RETURN c.category_name
)
FILTER "Restaurants" IN (
    FOR c IN 1..1 OUTBOUND b BusinessCategory
    RETURN c.category_name
)
RETURN b

Exécutée : True
Erreur : None
Nombre de résultats GENERATED : 0
Aperçu GENERATED : []
